# nb_slo_checks — freshness, volume and quality service levels (Snowflake)

Run by its task in tasks/monitor, tasks/privacy or tasks/admin. Edit the contracts, not this notebook.
> generated by: LakeLogic Lakehouse Studio
> source: contracts from lakelogic-rideflow-databricks-governed-lakehouse main

In [ ]:
# Parameters. A task passes them as one key=value string:
#   EXECUTE NOTEBOOK <DB>."pipeline"."<nb>"('key=value key=value')
# For an interactive run, edit the defaults below.
import sys
# ALL of sys.argv: Snowflake fills it from index 0 (no notebook name in front), so
# skipping index 0 drops the first argument. key=value parsing does not care where each sits.
_kv = dict(t.split('=', 1) for t in ' '.join(sys.argv).split() if '=' in t)
DOMAIN = _kv.get('domain', '')  # the domain whose _system.yaml to load
SYSTEM = _kv.get('system', '')  # the system within that domain
DB = _kv.get('db', '')  # the Snowflake database (the task passes it)
ENVIRONMENT = _kv.get('environment', 'dev')  # a key of _system.yaml environments
OP = _kv.get('op', 'slo')  # this notebook's job
SYSTEMS = _kv.get('systems', '')  # domain/system list, comma-separated (a job task passes every system)
FAIL_ON_BREACH = _kv.get('fail_on_breach', 'true')  # fail the task on a breach (slo, retention)
PIPELINE_RUN_ID = _kv.get('pipeline_run_id', '')  # optional link to the run that produced the data
print({'domain': DOMAIN, 'system': SYSTEM, 'db': DB, 'environment': ENVIRONMENT, 'op': OP, 'systems': SYSTEMS, 'fail_on_breach': FAIL_ON_BREACH, 'pipeline_run_id': PIPELINE_RUN_ID})

In [ ]:
# lakelogic[documents]==1.88.1: from the wheels staged in <DB>."pipeline"."wheels" when they hold LakeLogic
# (an account without PyPI access, e.g. a trial); else from PyPI. Only the wheels for THIS
# Python are copied: the whole stage (both Pythons, ~500 MB) was copied on every run, and
# PyPI was tried first and waited out where it is blocked (2026-11-01).
import os, subprocess, sys
def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *a]).returncode
from snowflake.snowpark.context import get_active_session
_s = get_active_session()
_v = f'cp{sys.version_info.major}{sys.version_info.minor}'
_staged = [r[0] for r in _s.sql(f'LIST @{DB}."pipeline"."wheels"').collect()]
if any('/lakelogic-' in f for f in _staged):
    os.makedirs('/tmp/wl', exist_ok=True)
    _s.file.get(f'@{DB}."pipeline"."wheels"', '/tmp/wl',
                pattern=f'.*-((py3|py2[.]py3)-none|{_v}-{_v}|cp3[0-9]+-abi3)-.*[.]whl')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index',
                    '--find-links', '/tmp/wl', 'lakelogic[documents]==1.88.1'], check=True)
else:
    if _pip('lakelogic[documents]==1.88.1', 'polars', 'pandas', 'pyarrow') != 0:
        raise RuntimeError('LakeLogic could not be installed: no wheels are staged in '
                           f'{DB}."pipeline"."wheels" and PyPI is not reachable.')

In [ ]:
# The registry paths are relative to the repository root; a task's CWD is not guaranteed to be it.
import os, sys
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'contracts')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, os.path.join(REPO, 'platform/snowflake/'.rstrip('/'))); os.chdir(REPO)
SYS = f'contracts/{DOMAIN}/{SYSTEM}/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB
from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection
session.sql(f'USE DATABASE {DB}').collect()
session.sql(f'USE SCHEMA {DB}."pipeline"').collect()
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)

In [ ]:
TARGETS = [f'contracts/{x.strip()}/_system.yaml' for x in str(SYSTEMS).split(',') if x.strip()] or [SYS]
print('systems:', TARGETS)

In [ ]:
from runtime import operations as ops
def _one(SYS):
    if OP == 'slo':
        return ops.slo_checks(conn, DB, SYS, environment=ENVIRONMENT,
                              fail_on_breach=ops._truthy(FAIL_ON_BREACH), pipeline_run_id=PIPELINE_RUN_ID or None)
    if OP == 'retention':
        return ops.retention_check(conn, DB, SYS, environment=ENVIRONMENT, fail_on_breach=ops._truthy(FAIL_ON_BREACH))
    if OP == 'maintenance':
        return ops.maintenance(conn, DB, SYS, environment=ENVIRONMENT)
    if OP == 'erasure':
        # Requests are ROWS in "pipeline"."erasure_requests" (tasks/privacy/_erasure.sql); a task
        # cannot take run-time arguments. Dry run unless the row says dry_run=false, confirm=ERASE
        # and carries a case_ref.
        out = []
        for req in session.sql(f'SELECT * FROM {DB}."pipeline"."erasure_requests" WHERE STATUS = \'pending\'').collect():
            out.append(ops.erasure(conn, DB, SYS, subject_column=req['SUBJECT_COLUMN'],
                subject_ids=[s.strip() for s in str(req['SUBJECT_IDS']).split(',') if s.strip()],
                environment=ENVIRONMENT, strategy=req['STRATEGY'] or 'nullify',
                dry_run=ops._truthy(req['DRY_RUN']), confirm=req['CONFIRM'] or '', case_ref=req['CASE_REF'] or ''))
        return out
    raise ValueError(f'unknown op {OP!r}: slo | retention | maintenance | erasure')

# One task runs the job for every system it names (Fabric: one pipeline per job). Every
# system is attempted; the task fails afterwards if any one did.
result, failed = {}, []
for SYS in TARGETS:
    try:
        result[SYS] = _one(SYS)
    except Exception as e:
        failed.append(f'{SYS}: {str(e)[:300]}')
        print(f'FAILED {SYS}: {e}')
if failed:
    raise RuntimeError(f'{len(failed)} of {len(TARGETS)} system(s) failed — ' + ' | '.join(failed))
result